# Travel Assistant Agent with AgentCore Long-Term Memory Strategies

## Introduction

You've experimented with AgentCore short-term memory fundamentals — creating memory resources, storing raw events, and building and an agent that maintains context within individual sessions.

Now we take the next step to enhance the intelligence of our agent: transforming those raw conversational events into structured, persistent insights using AgentCore's **long-term memory strategies**.

While short-term memory stores *what happened* in conversations, long-term memory extracts *what matters* — user preferences, key facts, and conversation summaries that persist across multiple sessions and interactions.

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Long-term Conversational Memory                                                  |
| Agent type          | Travel Planning Assistant (single agent)                                         |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                       |
| Tutorial components | Summary, Semantic, and User Preference memory strategies                         |
| Memory integration  | Hook-based (`MemorySessionManager` + `MemorySession` + custom `HookProvider`)   |
| Example complexity  | Intermediate                                                                     |

### What You Will Learn

- How to configure **all 3 long-term memory strategies** on a single memory resource
- Using `MemorySessionManager` and `MemorySession` for session-scoped memory operations
- Building a custom `HookProvider` to retrieve and inject long-term memories into agent context
- Inspecting what each strategy extracts from conversations
- Testing memory persistence across sessions

### Scenario

We'll build a **single Travel Agent** that leverages all three memory strategies:

1. **Summary Strategy** — Creates conversation summaries organized by topic (XML format)
2. **Semantic Strategy** — Stores factual information via vector embeddings for similarity search
3. **User Preference Strategy** — Tracks user-specific preferences in structured JSON

### Architecture

<div style="text-align:left">
    <img src="temp-sample-code/temp-riv-strategies-diagram.png" width="65%" />
</div>

### Prerequisites

- Python 3.10+
- AWS credentials with Amazon Bedrock AgentCore Memory permissions
- Amazon Bedrock AgentCore SDK

Let's get started!

## Step 1: Environment Setup

Let's begin by installing dependencies and importing the necessary libraries. But first, check that your kernel is set to Python3.

In [ ]:
%pip install -qU -r requirements.txt

In [ ]:
import logging
import time
from datetime import datetime
from strands import Agent
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole, RetrievalConfig

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("travel-assistant")

region = "us-west-2"
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

## Step 2: Create Memory Resource with 3 Long-Term Strategies

Let's configure **multiple memory strategies** to extract different types of insights from conversations.

Memory strategies must be defined at memory creation time and cannot be modified later.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

client = MemoryClient(region_name=region)

MEMORY_NAME = "TravelAssistantMemoryV2"

print("Creating or retrieving Memory with 3 Long-Term Strategies...")
memory = client.create_or_get_memory(
    name=MEMORY_NAME,
    description="Travel agent with all 3 long-term memory strategies",
    strategies=[
        {
            StrategyType.SUMMARY.value: {
                "name": "ConversationSummarizer",
                "description": "Summarizes travel conversations by topic",
                "namespaces": ["/travel/conversations/{actorId}/{sessionId}"]
            }
        },
        {
            StrategyType.SEMANTIC.value: {
                "name": "TravelFactExtractor",
                "description": "Stores factual travel information for similarity search",
                "namespaces": ["/travel/{actorId}/semantic"]
            }
        },
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "TravelPreferenceLearner",
                "description": "Captures user travel preferences over time",
                "namespaces": ["/travel/{actorId}/preferences"]
            }
        }
    ],
    event_expiry_days=7,
)

memory_id = memory["id"]
print(f"Memory ready: {memory_id}")

### Understanding the 3 Strategies

Each strategy processes raw conversational events differently:

| Strategy | What It Extracts | Output Format | Namespace Pattern |
|----------|-----------------|---------------|-------------------|
| **Summary** | Conversation summaries by topic | XML | `/travel/conversations/{actorId}/{sessionId}` |
| **Semantic** | Factual statements and details | Vector embeddings | `/travel/{actorId}/semantic` |
| **User Preference** | Preferences and personal patterns | Structured JSON | `/travel/{actorId}/preferences` |

**Asynchronous Processing:** Long-term memory extraction happens in the background. After events are stored, strategies typically process them within seconds to minutes. Your application continues operating while insights are being extracted.

## Step 3: Configure Memory Session and Hook Provider

We use `MemorySessionManager` and `MemorySession` from `bedrock_agentcore.memory.session` to manage session-scoped memory operations. Then we define a custom `HookProvider` that:
- **Retrieves** relevant long-term memories before each agent turn (on `MessageAddedEvent`)
- **Saves** conversation turns after each response (on `AfterInvocationEvent`)

This hook-based pattern gives explicit control over what context is injected and when interactions are persisted.

In [ ]:
ACTOR_ID = "user-001"
SESSION_ID = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"Actor ID: {ACTOR_ID}")
print(f"Session ID: {SESSION_ID}")

# Initialize the session manager and create a memory session
session_manager = MemorySessionManager(memory_id=memory_id, region_name=region)

travel_session = session_manager.create_memory_session(
    actor_id=ACTOR_ID,
    session_id=SESSION_ID,
)

print(f"Memory session created for actor: {ACTOR_ID}, session: {SESSION_ID}")

# Configure retrieval from each memory namespace (lower thresholds for better recall)
# Note: Summary strategy uses session-specific namespaces (/travel/conversations/{actorId}/{sessionId})
# which makes cross-session discovery impractical from the hook. Preferences and semantic
# strategies use actor-wide namespaces, so they work well for automatic retrieval.
retrieval_config = {
    "/travel/{actorId}/preferences/": RetrievalConfig(top_k=5, relevance_score=0.3),
    "/travel/{actorId}/semantic/": RetrievalConfig(top_k=3, relevance_score=0.2),
}


class TravelMemoryHookProvider(HookProvider):
    """Hook provider that retrieves long-term memories and saves interactions using MemorySession."""

    def __init__(self, memory_session: MemorySession, retrieval_config: dict = None):
        self.memory_session = memory_session
        self.retrieval_config = retrieval_config or {}

    def retrieve_memories(self, event: MessageAddedEvent):
        """Retrieve relevant long-term memories and inject them into the agent's system prompt."""
        messages = event.agent.messages
        if not messages or messages[-1]["role"] != "user":
            return
        # Skip tool result messages
        if "toolResult" in messages[-1].get("content", [{}])[0]:
            return

        user_query = messages[-1]["content"][0]["text"]
        all_memories = []

        try:
            for namespace_template, config in self.retrieval_config.items():
                resolved_namespace = namespace_template.format(
                    actorId=self.memory_session._actor_id
                )
                memories = self.memory_session.search_long_term_memories(
                    query=user_query,
                    namespace_prefix=resolved_namespace,
                    top_k=config.top_k,
                )
                # Filter by relevance score
                filtered = [
                    m for m in memories
                    if m.get("score", 0) >= config.relevance_score
                ]
                all_memories.extend(filtered)
                logger.info(
                    f"Retrieved {len(filtered)} memories from {resolved_namespace} "
                    f"(filtered from {len(memories)} total)"
                )

            if all_memories:
                context_lines = []
                for i, mem in enumerate(all_memories[:10], 1):
                    content = mem.get("content", {}).get("text", "")
                    score = mem.get("score", 0)
                    context_lines.append(f"{i}. (score: {score:.2f}) {content[:200]}")

                context_block = "\n".join(context_lines)
                event.agent.system_prompt = (
                    f"{event.agent.system_prompt}\n\n"
                    f"== Relevant memories from previous sessions ==\n{context_block}"
                )
                logger.info(f"Injected {len(all_memories)} memories into agent context")
            else:
                logger.info("No relevant memories found for this query")

        except Exception as e:
            logger.error(f"Failed to retrieve memories: {e}")

    def save_interaction(self, event: AfterInvocationEvent):
        """Save the last user/assistant turn to memory after each invocation."""
        try:
            messages = event.agent.messages
            if len(messages) < 2 or messages[-1]["role"] != "assistant":
                return

            # Find last user message and last assistant response
            customer_query = None
            agent_response = None

            for msg in reversed(messages):
                if msg["role"] == "assistant" and agent_response is None:
                    content = msg.get("content", [])
                    if content and isinstance(content[0], dict) and "text" in content[0]:
                        agent_response = content[0]["text"]
                elif msg["role"] == "user" and customer_query is None:
                    content = msg.get("content", [])
                    if content and isinstance(content[0], dict) and "text" in content[0]:
                        customer_query = content[0]["text"]
                        break

            if customer_query and agent_response:
                interaction_messages = [
                    ConversationalMessage(customer_query, MessageRole.USER),
                    ConversationalMessage(agent_response, MessageRole.ASSISTANT),
                ]
                result = self.memory_session.add_turns(interaction_messages)
                logger.info(f"Saved interaction to memory - Event ID: {result['eventId']}")

        except Exception as e:
            logger.error(f"Failed to save interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register memory retrieval and save hooks."""
        registry.add_callback(MessageAddedEvent, self.retrieve_memories)
        registry.add_callback(AfterInvocationEvent, self.save_interaction)
        logger.info("Travel memory hooks registered (retrieve on message, save after invocation)")


print("Session manager and TravelMemoryHookProvider configured.")

### How the Hook Provider Works

The `TravelMemoryHookProvider` registers two callbacks:

1. **`retrieve_memories` (on `MessageAddedEvent`)** — Fires when the user sends a message. It searches all configured namespaces using `session.search_long_term_memories()`, filters results by relevance score, and appends the matching memories to the agent's system prompt so the model sees them as context.

2. **`save_interaction` (on `AfterInvocationEvent`)** — Fires after the agent responds. It extracts the last user/assistant turn and saves it via `session.add_turns()`, which triggers asynchronous strategy processing.

The `retrieval_config` controls what gets retrieved:
- **Preferences namespace** (`top_k=5, relevance_score=0.3`): Retrieves up to 5 stored preferences
- **Semantic namespace** (`top_k=3, relevance_score=0.2`): Retrieves up to 3 factual memories matching the query

Lower relevance thresholds (0.2-0.3) ensure useful memories aren't filtered out too aggressively.

**Note on Summary strategy:** Summaries are stored at session-specific namespaces (`/travel/conversations/{actorId}/{sessionId}`), which requires knowing exact session IDs to query. We query these directly in Step 8 for educational purposes, but the hook focuses on the actor-wide preferences and semantic namespaces for automatic retrieval.

## Step 4: Add Observability Hook

In addition to the memory hook, we add an observability hook that logs interaction details — useful for monitoring, debugging, and metrics. This demonstrates how multiple hook providers can be composed on a single agent.

In [ ]:
class ObservabilityHookProvider(HookProvider):
    """Hook for monitoring agent interactions — demonstrates hooks alongside session manager."""

    def log_interaction(self, event: AfterInvocationEvent):
        """Log interaction details for observability."""
        messages = event.agent.messages
        turn_count = len([m for m in messages if m["role"] == "user"])
        last_assistant = None
        for msg in reversed(messages):
            if msg["role"] == "assistant":
                content = msg.get("content", [])
                if content and isinstance(content[0], dict) and "text" in content[0]:
                    last_assistant = content[0]["text"][:80]
                    break
        logger.info(f"Turn {turn_count} completed | Response preview: {last_assistant}...")

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register observability hooks."""
        registry.add_callback(AfterInvocationEvent, self.log_interaction)
        logger.info("Observability hooks registered")

### Hooks for Different Concerns

| Concern | Mechanism |
|---------|----------|
| Memory retrieval & injection | `TravelMemoryHookProvider` (on `MessageAddedEvent`) |
| Memory save (persistence) | `TravelMemoryHookProvider` (on `AfterInvocationEvent`) |
| Observability, logging, metrics | `ObservabilityHookProvider` (custom behavior) |

This notebook uses hooks for both memory management and observability. Multiple hook providers can be composed together — the agent calls each registered callback at the appropriate lifecycle point.

## Step 5: Create the Travel Agent

Now we assemble our single Travel Agent with the memory hook provider for retrieval/save and the observability hook for logging.

In [ ]:
TRAVEL_AGENT_PROMPT = """You are a travel planning assistant. You help customers plan trips,
find flights, book hotels, and provide travel advice. You are friendly, concise, and helpful.

Use what you know about the user's preferences from memory to make personalized recommendations.
When you recall something about the user from a previous session, mention it naturally in conversation.

You do not need to ask for identification information — assume user details are already known.
Keep responses clear and concise.
"""

# Create the hook provider instance with our session and retrieval config
memory_hooks = TravelMemoryHookProvider(travel_session, retrieval_config)

travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_PROMPT,
    model=MODEL_ID,
    hooks=[memory_hooks, ObservabilityHookProvider()],
)

print("Travel agent created with memory hooks and observability hook.")

## Step 6: Seed Conversation Data

Before interacting with the agent, let's seed the memory with a previous travel conversation. This simulates a prior session where the user discussed travel preferences — the strategies will extract insights from this data.

We use `session.add_turns()` with `ConversationalMessage` objects to store conversation history, which triggers asynchronous processing by all 3 strategies.

In [ ]:
# Previous conversation: user discussing a NY to London trip
# We use a separate session for seeding so it doesn't mix with the live session
seed_session_id = "seed-session-nyc-london"
seed_session = session_manager.create_memory_session(
    actor_id=ACTOR_ID,
    session_id=seed_session_id,
)

seed_messages = [
    ConversationalMessage("Hi, I'm looking to book a flight from New York to London sometime next month.", MessageRole.USER),
    ConversationalMessage("I'd be happy to help you find flights from New York to London! Do you have specific dates in mind, or are you flexible?", MessageRole.ASSISTANT),
    ConversationalMessage("I'm thinking around the 15th to the 25th, but I can be a bit flexible.", MessageRole.USER),
    ConversationalMessage("Great! Do you have any preferences regarding airlines or flight times?", MessageRole.ASSISTANT),
    ConversationalMessage("I definitely prefer direct flights if possible. I really don't like layovers.", MessageRole.USER),
    ConversationalMessage("I completely understand. There are several airlines offering direct flights between New York and London, including British Airways, American Airlines, Delta, and Virgin Atlantic.", MessageRole.ASSISTANT),
    ConversationalMessage("I've had good experiences with British Airways in the past.", MessageRole.USER),
    ConversationalMessage("British Airways offers excellent service on transatlantic routes. Do you have seating preferences?", MessageRole.ASSISTANT),
    ConversationalMessage("I always try to get an aisle seat. I like being able to get up without disturbing others, especially on long flights.", MessageRole.USER),
    ConversationalMessage("An aisle seat is great for long-haul flights. Would you prefer morning, afternoon, or evening departures?", MessageRole.ASSISTANT),
    ConversationalMessage("I prefer overnight flights for long journeys. It helps me adjust to the time difference better.", MessageRole.USER),
    ConversationalMessage("Overnight flights are indeed smart for eastbound transatlantic travel — you arrive in London in the morning and minimize jet lag. British Airways and Delta both offer evening departures.", MessageRole.ASSISTANT),
    ConversationalMessage("Perfect! And I usually stay at boutique hotels rather than big chains. I like places with character.", MessageRole.USER),
    ConversationalMessage("Boutique hotels are wonderful for experiencing the local culture! London has many great options in neighborhoods like Covent Garden, Shoreditch, and South Kensington. Any budget range in mind?", MessageRole.ASSISTANT),
    ConversationalMessage("I'd say mid-range — around $200-300 per night. And I always want breakfast included if possible.", MessageRole.USER),
    ConversationalMessage("That's a great budget for boutique hotels in London. Many in that range include breakfast. I'll note your preferences for future recommendations!", MessageRole.ASSISTANT),
]

print("Seeding memory with previous travel conversation...")
result = seed_session.add_turns(seed_messages)
print(f"Conversation seeded (session: {seed_session_id}, event ID: {result['eventId']})")

In [ ]:
# Wait for asynchronous strategy processing
# The 3 strategies need time to extract insights from the seeded conversation.
# This may take a seconds to minutes depending on conversation length.
print("Waiting for long-term memory strategies to process the seeded data...")
time.sleep(30)
print("Processing complete. Long-term memories should now be available.")

## Step 7: Interact with the Agent

Now let's have a multi-turn conversation with our agent. The memory hook will automatically:
- Retrieve relevant long-term memories (preferences, facts, summaries) before each turn
- Save new conversation turns after each response

Watch how the agent uses knowledge from the seeded conversation to personalize its responses!

In [ ]:
travel_agent("Hello! I'd like to plan a trip from Seattle to Berlin for about 2 weeks in July.")

In [ ]:
travel_agent("I prefer boutique hotels when I travel. Can you recommend a hotel in central Berlin?")

## Step 8: Explore Long-Term Memory Retrieval

Let's look under the hood at what each strategy extracted from our conversations. We'll query each namespace directly using `client.retrieve_memories()` to see the raw outputs.

### What Each Strategy Produces

- **User Preferences**: Structured JSON capturing likes, dislikes, and patterns (e.g., `{"preference": "aisle seat", "category": "seating"}`)
- **Semantic Facts**: Factual statements stored as embeddings for similarity search (e.g., "User is planning a trip from LA to Madrid in July")
- **Summaries**: XML-formatted conversation summaries organized by topic

In [ ]:
# Retrieve user preferences extracted by the User Preference strategy
print("=" * 60)
print("USER PREFERENCES (structured JSON)")
print("=" * 60)

preferences = client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"/travel/{ACTOR_ID}/preferences",
    query="travel preferences",
)

for i, mem in enumerate(preferences, 1):
    content = mem.get("content", {})
    if isinstance(content, dict) and "text" in content:
        print(f"  {i}. {content['text']}")
    else:
        print(f"  {i}. {content}")

In [ ]:
# Retrieve semantic facts extracted by the Semantic strategy
print("=" * 60)
print("SEMANTIC FACTS (vector embeddings)")
print("=" * 60)

facts = client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"/travel/{ACTOR_ID}/semantic",
    query="flight booking details and travel plans",
)

for i, mem in enumerate(facts, 1):
    content = mem.get("content", {})
    if isinstance(content, dict) and "text" in content:
        print(f"  {i}. {content['text']}")
    else:
        print(f"  {i}. {content}")

In [ ]:
# Retrieve conversation summaries from the Summary strategy
# Note: Summaries are stored at the session-specific namespace (includes sessionId),
# so we must query with the exact session namespace used during seeding.
print("=" * 60)
print("CONVERSATION SUMMARIES (XML format)")
print("=" * 60)

summaries = client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"/travel/conversations/{ACTOR_ID}/{seed_session_id}",
    query="travel plans and trip details",
)

for i, mem in enumerate(summaries, 1):
    content = mem.get("content", {})
    if isinstance(content, dict) and "text" in content:
        print(f"  {i}. {content['text'][:200]}...")
    else:
        print(f"  {i}. {content}")

## Step 9: Test Memory Persistence Across Sessions

The real power of long-term memory: insights persist across sessions. Let's create a **brand new session** with the same actor and verify the agent still remembers preferences from prior conversations.

In [ ]:
# Create a completely new session to test cross-session memory persistence
NEW_SESSION_ID = f"travel-session-new-{datetime.now().strftime('%Y%m%d%H%M%S')}"
print(f"New session: {NEW_SESSION_ID}")

# Create a new MemorySession with the same actor but a different session ID
new_travel_session = session_manager.create_memory_session(
    actor_id=ACTOR_ID,
    session_id=NEW_SESSION_ID,
)

# Create a new hook provider instance pointing to the new session
new_memory_hooks = TravelMemoryHookProvider(new_travel_session, retrieval_config)

# Create a fresh agent instance with the new hooks
new_travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_PROMPT,
    model=MODEL_ID,
    hooks=[new_memory_hooks, ObservabilityHookProvider()],
)

print("New agent created in a fresh session. Let's test memory persistence...")
new_travel_agent("Hi! Can you remind me what my travel preferences are? I've told you before.")

## Summary

In this lab, you've built a Travel Agent with **all 3 AgentCore long-term memory strategies** using a hook-based memory integration:

| Strategy | What It Did |
|----------|------------|
| **Summary** | Created topic-organized summaries of conversations |
| **Semantic** | Stored factual details as searchable embeddings |
| **User Preference** | Extracted structured preference data for personalization |

### Key Takeaways

1. **`TravelMemoryHookProvider`** gives explicit control over memory retrieval and injection — memories are searched via `MemorySession.search_long_term_memories()` and appended to the system prompt
2. **`MemorySessionManager` + `MemorySession`** provide a clean session-scoped API for both reads and writes
3. **Hooks fire at the right lifecycle points** — `MessageAddedEvent` for retrieval (before the model runs), `AfterInvocationEvent` for persistence (after the model responds)
4. **Long-term strategies process asynchronously** — raw events are stored immediately, insights are extracted in 10–30 seconds
5. **Memories persist across sessions** — same actor, different session = full preference recall
6. **Namespace design matters** — actor-wide paths enable cross-session discovery, session-specific paths organize by interaction

## Cleanup

Delete the memory resource to clean up resources used in this notebook.

In [ ]:
# Uncomment to delete the memory resource:
# client.delete_memory_and_wait(
#     memory_id=memory_id,
#     max_wait=300,
#     poll_interval=10,
# )
# print(f"Memory {memory_id} deleted.")